# Periodic residuals for equirectangular bundle adjustment

An equirectangular image joins its left and right edges. Its horizontal bundle-adjustment residual must therefore be periodic: a projection at pixel 638 and an observation at pixel 2 in a 640-pixel image are four pixels apart, not 636.

This notebook demonstrates the seam case, runs Spirula's CPU/Vulkan regression when a matching build is present, and provides a native CLI harness for a real COLMAP sparse model. It is a hand-run reference and is not imported or required by the build.

In [ ]:
def periodic_residual(projected_x: float, observed_x: float, width: float) -> float:
    residual = projected_x - observed_x
    half = 0.5 * width
    if residual > half:
        residual -= width
    elif residual < -half:
        residual += width
    return residual

width = 640.0
examples = [(638.0, 2.0), (2.0, 638.0)]
for projected, observed in examples:
    naive = projected - observed
    wrapped = periodic_residual(projected, observed, width)
    print(f"projected={projected:5.1f} observed={observed:5.1f}  naive={naive:6.1f} wrapped={wrapped:4.1f}")

Expected output:

```text
projected=638.0 observed=  2.0  naive= 636.0 wrapped=-4.0
projected=  2.0 observed=638.0  naive=-636.0 wrapped= 4.0
```

The same branch is applied to the autodifferentiated residual, so the selected periodic branch and its Jacobian agree on both host and Vulkan paths. Other camera models retain their ordinary non-periodic residuals.

In [ ]:
from pathlib import Path
import os
import subprocess

root = Path.cwd()
if not (root / "CMakeLists.txt").is_file():
    root = Path.cwd().resolve().parents[1]

test_candidates = [
    root / "build" / "sfm_equirect_seam_test.exe",
    root / "build" / "sfm_equirect_seam_test",
]
seam_test = next((path for path in test_candidates if path.is_file()), None)
if seam_test is None:
    print("Build with SS_BUILD_SFM=ON; sfm_equirect_seam_test was not found.")
else:
    completed = subprocess.run([str(seam_test)], cwd=root, text=True, capture_output=True)
    print(completed.stdout, end="")
    print(completed.stderr, end="")
    completed.check_returncode()

On an RTX 4080, the regression places 14 of 48 observations across the seam and compares CPU against native Vulkan for cost, the assembled Schur system, gradient, final cost, poses and points. The complete float/double and trivial/Huber/Cauchy matrix reports:

```text
float    trivial  cost 4.93e-06  S 2.22e-06  g 3.22e-06  final 1.23e-08  pose 1.51e-03  point 1.98e-04  PASS
double   trivial  cost 1.00e-06  S 8.07e-08  g 4.96e-07  final 3.71e-12  pose 2.98e-07  point 1.36e-06  PASS
float    huber    cost 4.02e-06  S 3.24e-06  g 1.98e-06  final 9.60e-08  pose 3.85e-04  point 5.96e-04  PASS
double   huber    cost 7.98e-07  S 5.88e-07  g 1.27e-07  final 3.71e-12  pose 2.97e-07  point 1.36e-06  PASS
float    cauchy   cost 3.23e-06  S 4.96e-06  g 1.43e-06  final 0.00e+00  pose 4.62e-03  point 5.70e-04  PASS
double   cauchy   cost 6.25e-07  S 7.33e-07  g 1.31e-07  final 3.71e-12  pose 2.97e-07  point 1.36e-06  PASS
PASS
```

For the two-point demonstration above, the unfixed objective is approximately 404,000. The remaining CPU/device differences are bounded by the Vulkan projection's `atan2` approximation.

In [ ]:
def find_sfm_executable() -> Path | None:
    candidates = [
        root / "build" / "spirula-sfm.exe",
        root / "build" / "spirula-sfm",
        root / "build" / "spirula.exe",
        root / "build" / "spirula",
    ]
    return next((path for path in candidates if path.is_file()), None)

def run_native_ba(executable: Path, model: Path) -> str:
    env = os.environ.copy()
    env.pop("SS_SFM_BA_GEODESIC", None)
    command = [str(executable)]
    if executable.stem == "spirula":
        command.append("sfm")
    command.extend([
        "ba", str(model),
        "--real", "double", "--loss", "trivial", "--solver", "dense",
        "--max-iters", "80", "--damping", "1e-8",
        "--rtol", "1e-10", "--patience", "20", "--quiet",
    ])
    completed = subprocess.run(command, cwd=root, env=env, text=True, capture_output=True)
    output = completed.stdout + completed.stderr
    if completed.returncode:
        raise RuntimeError(output)
    return output

model_text = os.environ.get("SS_EQUIRECT_MODEL", "")
model = Path(model_text) if model_text else None
executable = find_sfm_executable()
if executable is None or model is None:
    print("Set SS_EQUIRECT_MODEL to a COLMAP sparse-model directory and build the SfM CLI.")
else:
    print(run_native_ba(executable, model))

## Worked real-capture result

The fix was evaluated on a ten-frame, 4096×2048 equirectangular capture. The sparse test model contained 9 registered images, 1,441 points and 3,148 observations. Images are not redistributed with the repository. The cell above reproduces the native solve on any equivalent COLMAP model supplied through `SS_EQUIRECT_MODEL`.

| residual | initial cost | final cost | LM iterations |
|---|---:|---:|---:|
| non-periodic | 141,390,489 | 141,335,658 | 80 |
| periodic | 18,076,110 | 1,062 | 39 |

The full reconstruction registered 9/10 images with 1,441 points and mean/median reprojection errors of 0.532/0.337 pixels. The improvement comes from correcting the objective at the panorama boundary; no alternative solver or camera-ray convention is introduced.

## Build used for the native checks

```bat
build_develop.bat ^
  -DSS_BACKEND=vulkan ^
  -DSS_BUILD_CLI=ON ^
  -DSS_BUILD_GUI=OFF ^
  -DSS_BUILD_SFM=ON ^
  -DSS_BUILD_SAM=OFF ^
  -DSS_SEPARATE_TOOLS=OFF ^
  "-DSS_SFM_REALS=float;double" ^
  "-DSS_SFM_LOSSES=trivial;huber;cauchy"
```

The same executable evaluates the host implementation and each built Vulkan variant before comparing their intermediate and final results.